
# AI Cinematic Scene-to-Score Generator — Unified Version

This notebook combines all the ideas into one system:

- **emotion detection** from text
- **scene understanding** for film-style scoring
- **cinematic prompt generation** for MusicGen
- **audio generation**
- **symbolic score generation** with `music21`
- **MIDI + MusicXML export** so a musician can reuse the result
- **rough emotional feedback loop** to refine the prompt when needed

The goal is not only to generate audio, but to create a **composer inspiration draft** that can be listened to, edited, and developed into a real soundtrack.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
%cd /content
!rm -rf Emotion-Driven-Music-Generator

/content


In [14]:
!git clone https://github.com/kyrie11-haoran/Emotion-Driven-Music-Generator.git
%cd /content/Emotion-Driven-Music-Generator

Cloning into 'Emotion-Driven-Music-Generator'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 38 (delta 15), reused 7 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 3.14 MiB | 11.30 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/Emotion-Driven-Music-Generator


In [16]:
!ls "/content/drive/MyDrive/Colab Notebooks"

 AAT.ipynb
'AAT_Lab1 (1).ipynb'
 AAT_Lab1.ipynb
'AAT_Lab2 (1).ipynb'
 AAT_Lab2.ipynb
 AAT_Lab3.ipynb
 AAT_Lab4.ipynb
'AAT_Lab5(1).ipynb'
 AAT_Lab6.ipynb
 AI_Cinematic_Scene_to_Score_Generator.ipynb
'Another copy of AAT_Lab5(1).ipynb'
 Assessment_1.ipynb
 Assessment2.ipynb
 Assessment3.ipynb
 assessment3_retrieval_data.csv
 Cinematic_Scene_to_Score_Generator.ipynb
 CMAA_0
 CMAA_A1.ipynb
 CMAA_Lab1.ipynb
'Copia de 4c16-git-workflow.ipynb'
'Copia de 4c16-init.ipynb'
'Copia de Te damos la bienvenida a Colaboratory'
'Copy of AAT_Lab5(1).ipynb'
 emotion_music.ipynb
 media
 Untitled
 Untitled0.ipynb
'Untitled (1)'
 Untitled1.ipynb


In [7]:
!git add AI_Cinematic_Scene_to_Score_Generator_Unified.ipynb
!git commit -m "Add unified cinematic scene-to-score notebook"
!git push origin main

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [ ]:
!pip install transformers torch torchaudio scipy gradio accelerate music21 -q

In [ ]:

import os
import re
import uuid
import random
import shutil
import zipfile
import tempfile
from pathlib import Path

import numpy as np
import torch
import gradio as gr
from music21 import stream, note, chord, tempo, meter, key, scale, instrument
from transformers import pipeline, MusicgenForConditionalGeneration, AutoProcessor

print("Loading models...")

emotion_classifier = pipeline(
    "text-classification",
    model="bhadresh-savani/distilbert-base-uncased-emotion",
    top_k=None
)

musicgen_processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
musicgen_model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small")

device = "cuda" if torch.cuda.is_available() else "cpu"
musicgen_model.to(device)

OUTPUT_DIR = Path("scene_to_score_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"✅ Models loaded on {device}")


Loading models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

✅ Models loaded on cpu


In [ ]:

# -------------------------------------------------
# CINEMATIC KNOWLEDGE BASE
# -------------------------------------------------

SCENE_KEYWORDS = {
    "romance": ["love", "kiss", "embrace", "together", "heart", "romantic"],
    "suspense": ["dark", "alone", "afraid", "shadow", "fear", "silence", "corridor"],
    "action": ["run", "fight", "escape", "chase", "explosion", "attack", "sprint"],
    "memory": ["remember", "past", "childhood", "used to", "back then", "miss"],
    "tragedy": ["lost", "death", "goodbye", "pain", "grief", "cry", "funeral"],
    "mystery": ["secret", "unknown", "clue", "hidden", "discovered", "question"],
    "triumph": ["won", "victory", "finally", "achieved", "success", "dream came true"],
}

SCENE_ARCS = {
    "suspense": "begin with sparse texture and low drones, slowly build tension, reach a dark climax, end unresolved",
    "romance": "start with intimate piano, add warm strings, grow into an emotional peak, resolve gently",
    "action": "open with rhythmic pulse, build with percussion and fast strings, hit an explosive climax, finish sharply",
    "memory": "start fragile and nostalgic, add soft pads and reflective harmony, rise with warmth, fade delicately",
    "tragedy": "open with slow strings and heavy atmosphere, deepen emotionally, collapse quietly at the end",
    "mystery": "start minimal and curious, add subtle tension, slowly reveal layers, finish with uncertainty",
    "triumph": "open with hopeful motion, broaden harmonically, peak with uplifting energy, end confidently",
    "drama": "start emotionally restrained, develop layered harmony, reach a cinematic peak, resolve softly",
}

LEITMOTIFS = {
    "hero": "noble rising string motif",
    "villain": "dark low brass motif",
    "memory": "fragile piano motif",
    "love": "warm lyrical violin motif",
    "none": ""
}

EMOTION_TO_SCORE = {
    "joy":      {"tempo": 120, "key": "C", "mode": "major", "instrument": "piano",   "register": ("C4", "C6")},
    "sadness":  {"tempo": 62,  "key": "A", "mode": "minor", "instrument": "piano",   "register": ("A3", "A5")},
    "fear":     {"tempo": 78,  "key": "D", "mode": "minor", "instrument": "violin",  "register": ("D4", "D6")},
    "anger":    {"tempo": 132, "key": "E", "mode": "minor", "instrument": "cello",   "register": ("E3", "E5")},
    "love":     {"tempo": 84,  "key": "G", "mode": "major", "instrument": "violin",  "register": ("G3", "G5")},
    "surprise": {"tempo": 110, "key": "D", "mode": "major", "instrument": "flute",   "register": ("D4", "D6")},
}

GENRE_INSTRUMENT_BIAS = {
    "drama": "piano and strings",
    "horror": "drones, low strings and atmospheric textures",
    "romance": "piano, strings and warm pads",
    "thriller": "pulses, low piano and tense strings",
    "sci-fi": "synth pads, pulses and atmospheric layers",
    "fantasy": "harp, strings and woodwinds"
}


In [ ]:

# -------------------------------------------------
# ANALYSIS + PROMPT HELPERS
# -------------------------------------------------

def normalize_emotions(emotions):
    return sorted(emotions, key=lambda x: x["score"], reverse=True)

def get_emotions(text):
    emotions = emotion_classifier(text)[0]
    return normalize_emotions(emotions)

def detect_scene_type(text):
    text_lower = text.lower()
    scores = {scene: sum(word in text_lower for word in words) for scene, words in SCENE_KEYWORDS.items()}
    best_scene = max(scores, key=scores.get)
    return best_scene if scores[best_scene] > 0 else "drama"

def split_into_sentences(text):
    return [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]

def emotion_journey(text):
    lines = []
    for idx, sentence in enumerate(split_into_sentences(text), start=1):
        e = get_emotions(sentence)[0]
        lines.append(f"{idx}. {e['label']} ({e['score']:.1%}) → {sentence}")
    return "\n".join(lines) if lines else "1. neutral → single short text"

def build_cinematic_prompt(text, emotions, scene, genre, intensity, ending, leitmotif):
    top1 = emotions[0]["label"]
    top2 = emotions[1]["label"] if len(emotions) > 1 else top1
    arc = SCENE_ARCS.get(scene, SCENE_ARCS["drama"])
    motif = LEITMOTIFS.get(leitmotif, "")
    genre_bias = GENRE_INSTRUMENT_BIAS.get(genre, "cinematic instrumentation")

    prompt = (
        f"{genre} film soundtrack, {scene} scene, intensity {intensity}/10, "
        f"blending {top1} and {top2}, {genre_bias}, {arc}, {ending} ending, "
        f"{motif}, expressive instrumental cue, suitable for background score"
    )
    return prompt

def build_cue_title(scene, genre, emotions):
    top = emotions[0]["label"].title()
    return f"{scene.title()} Cue in {genre.title()} — {top}"

def build_score_brief(scene, genre, emotions, prompt):
    blend = " + ".join([e["label"] for e in emotions[:2]])
    return f'''
🎬 **Cue Title:** {build_cue_title(scene, genre, emotions)}

🎭 **Emotion Blend:** {blend}

🎥 **Scene Type:** {scene}

🎼 **Scoring Use:** Composer inspiration draft for a {genre} film cue

🎵 **Generation Prompt:** {prompt}
'''.strip()


In [ ]:

# -------------------------------------------------
# AUDIO FEEDBACK LOOP
# -------------------------------------------------

def extract_audio_features(audio):
    audio = np.asarray(audio).flatten()
    energy = float(np.mean(np.abs(audio)))
    zcr = float(np.mean(np.abs(np.diff(np.sign(audio))) > 0))
    variance = float(np.var(audio))
    return {"energy": energy, "zcr": zcr, "variance": variance}

def estimate_emotion_from_audio(features):
    # Very rough heuristic: simple enough for a student demo
    if features["energy"] < 0.015:
        return "sadness"
    if features["energy"] > 0.045 and features["zcr"] > 0.08:
        return "anger"
    if features["energy"] > 0.035:
        return "joy"
    if features["zcr"] > 0.09:
        return "fear"
    return "sadness"

def refine_prompt(prompt, target_emotion, detected_emotion):
    if target_emotion == detected_emotion:
        return prompt
    if target_emotion == "sadness":
        return prompt + ", slower tempo, softer dynamics, more fragile and reflective"
    if target_emotion == "joy":
        return prompt + ", brighter harmony, more lift, more energetic pulse"
    if target_emotion == "fear":
        return prompt + ", darker texture, more suspense, more tension and instability"
    if target_emotion == "anger":
        return prompt + ", stronger percussion, heavier accents, more aggressive momentum"
    if target_emotion == "love":
        return prompt + ", warmer strings, tenderness, smoother phrasing"
    return prompt + ", clearer emotional character"


In [ ]:

# -------------------------------------------------
# SYMBOLIC SCORE GENERATION (MIDI + MUSICXML)
# -------------------------------------------------

def _make_scale(root, mode):
    return scale.MajorScale(root) if mode == "major" else scale.MinorScale(root)

def _pick_instrument(name):
    name = name.lower()
    if name == "violin":
        return instrument.Violin()
    if name == "cello":
        return instrument.Violoncello()
    if name == "flute":
        return instrument.Flute()
    return instrument.Piano()

def symbolic_parameters(primary_emotion, scene, genre, intensity):
    base = EMOTION_TO_SCORE.get(primary_emotion, EMOTION_TO_SCORE["sadness"]).copy()

    if scene == "action":
        base["tempo"] += 18
    elif scene == "suspense":
        base["tempo"] -= 8
    elif scene == "triumph":
        base["tempo"] += 10

    base["tempo"] = max(45, min(150, int(base["tempo"] + (intensity - 5) * 4)))

    if genre in ["horror", "thriller"]:
        base["mode"] = "minor"
    elif genre == "romance" and primary_emotion != "fear":
        base["mode"] = "major"

    return base

def generate_symbolic_score(emotions, scene, genre, intensity, duration_bars=8):
    primary = emotions[0]["label"]
    params = symbolic_parameters(primary, scene, genre, intensity)

    sc = _make_scale(params["key"], params["mode"])
    pitches = sc.getPitches(params["register"][0], params["register"][1])

    # -----------------------------
    # Create parts
    # -----------------------------
    melody = stream.Part()
    melody.append(_pick_instrument(params["instrument"]))
    melody.append(tempo.MetronomeMark(number=params["tempo"]))
    melody.append(meter.TimeSignature('4/4'))
    melody.append(key.Key(params["key"], params["mode"]))

    harmony = stream.Part()
    harmony.append(instrument.Piano())
    harmony.append(tempo.MetronomeMark(number=params["tempo"]))
    harmony.append(meter.TimeSignature('4/4'))
    harmony.append(key.Key(params["key"], params["mode"]))

    bass = stream.Part()
    bass.append(instrument.Violoncello())
    bass.append(tempo.MetronomeMark(number=params["tempo"]))
    bass.append(meter.TimeSignature('4/4'))
    bass.append(key.Key(params["key"], params["mode"]))

    # -----------------------------
    # Better chord progressions
    # -----------------------------
    if params["mode"] == "major":
        progression = [
            [0, 2, 4],   # I
            [3, 5, 0],   # IV
            [4, 6, 1],   # V
            [0, 2, 4]    # I
        ]
    else:
        progression = [
            [0, 2, 4],   # i
            [5, 0, 2],   # VI
            [3, 5, 0],   # III
            [4, 6, 1]    # VII
        ]

    # -----------------------------
    # Melody pattern ideas
    # -----------------------------
    motif_intervals = {
        "joy": [0, 2, 4, 7],
        "sadness": [0, 2, 3, 5],
        "fear": [0, 1, 3, 6],
        "anger": [0, 3, 5, 7],
        "love": [0, 2, 4, 5],
        "surprise": [0, 2, 4, 6],
    }.get(primary, [0, 2, 3, 5])

    root_index = len(pitches) // 3

    # Keep a tiny repeated motif so it sounds less random
    base_motif = []
    for _ in range(4):
        step = random.choice(motif_intervals)
        idx = min(max(root_index + step, 0), len(pitches) - 1)
        base_motif.append(pitches[idx])

    # -----------------------------
    # Build bars
    # -----------------------------
    for bar in range(duration_bars):
        triad = progression[bar % len(progression)]

        # --- Harmony track: one chord per bar
        chord_pitches = []
        for i in triad:
            if i < len(pitches):
                chord_pitches.append(pitches[i])

        h = chord.Chord(chord_pitches, quarterLength=4.0)
        h.volume.velocity = random.randint(45, 65)
        harmony.append(h)

        # --- Bass track: root note pulse
        bass_pitch = chord_pitches[0]
        for beat in range(2):
            b = note.Note(bass_pitch, quarterLength=2.0)
            b.octave = max(2, b.octave - 1)
            b.volume.velocity = random.randint(50, 75)
            bass.append(b)

        # --- Melody track
        beats_remaining = 4.0
        motif_position = 0

        while beats_remaining > 0:
            # Better rhythm choices: more musical, less chaotic
            dur = random.choice([0.5, 1.0, 1.0, 1.0, 2.0])
            dur = min(dur, beats_remaining)

            # Some rests for memory/tragedy scenes
            if scene in ["memory", "tragedy"] and random.random() < 0.12:
                n = note.Rest(quarterLength=dur)
            else:
                # Reuse motif often, but with slight variation
                if random.random() < 0.7:
                    pitch_choice = base_motif[motif_position % len(base_motif)]
                    motif_position += 1
                else:
                    step = random.choice(motif_intervals)
                    idx = min(max(root_index + step, 0), len(pitches) - 1)
                    pitch_choice = pitches[idx]

                n = note.Note(pitch_choice, quarterLength=dur)
                n.volume.velocity = random.randint(55, 90)

            melody.append(n)
            beats_remaining -= dur

    # -----------------------------
    # Combine score
    # -----------------------------
    score = stream.Score()
    score.insert(0, melody)
    score.insert(0, harmony)
    score.insert(0, bass)

    # Preview text
    note_preview = []
    for n in melody.recurse().notes[:12]:
        note_preview.append(f"{n.pitch.nameWithOctave} ({n.quarterLength})")

    return score, params, ", ".join(note_preview[:12])

def export_symbolic_files(score, stem_name):
    midi_path = OUTPUT_DIR / f"{stem_name}.mid"
    xml_path = OUTPUT_DIR / f"{stem_name}.musicxml"
    score.write("midi", fp=str(midi_path))
    score.write("musicxml", fp=str(xml_path))
    return str(midi_path), str(xml_path)


In [ ]:

# -------------------------------------------------
# MUSICGEN AUDIO GENERATION
# -------------------------------------------------

def generate_audio_from_prompt(prompt, duration, temperature):
    inputs = musicgen_processor(text=[prompt], padding=True, return_tensors="pt").to(device)
    max_new_tokens = int(duration * 50)

    with torch.no_grad():
        audio_values = musicgen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            guidance_scale=3.0
        )

    sampling_rate = musicgen_model.config.audio_encoder.sampling_rate
    audio_data = audio_values[0, 0].cpu().numpy()
    return sampling_rate, audio_data

def generate_with_feedback(prompt, target_emotion, duration, temperature, use_feedback=True):
    sr, audio = generate_audio_from_prompt(prompt, duration, temperature)
    detected = "not checked"
    refined_prompt = prompt
    features = extract_audio_features(audio)

    if use_feedback:
        detected = estimate_emotion_from_audio(features)
        if detected != target_emotion:
            refined_prompt = refine_prompt(prompt, target_emotion, detected)
            sr, audio = generate_audio_from_prompt(refined_prompt, duration, temperature)
            features = extract_audio_features(audio)
            detected = estimate_emotion_from_audio(features)

    return sr, audio, features, detected, refined_prompt


In [ ]:

# -------------------------------------------------
# MAIN PIPELINE
# -------------------------------------------------

def generate_scene_to_score(
    text,
    duration=10,
    temperature=1.0,
    genre="drama",
    intensity=5,
    ending="soft fade",
    leitmotif="none",
    use_feedback=True
):
    if not text or not text.strip():
        raise ValueError("Please enter a scene description.")

    emotions = get_emotions(text)
    scene = detect_scene_type(text)
    prompt = build_cinematic_prompt(text, emotions, scene, genre, intensity, ending, leitmotif)

    sr, audio, features, detected_audio_emotion, final_prompt = generate_with_feedback(
        prompt=prompt,
        target_emotion=emotions[0]["label"],
        duration=duration,
        temperature=temperature,
        use_feedback=use_feedback
    )

    score, params, note_preview = generate_symbolic_score(
        emotions=emotions,
        scene=scene,
        genre=genre,
        intensity=intensity,
        duration_bars=max(4, min(12, duration // 2))
    )

    stem_name = f"cue_{uuid.uuid4().hex[:8]}"
    midi_path, xml_path = export_symbolic_files(score, stem_name)

    bundle_path = OUTPUT_DIR / f"{stem_name}_bundle.zip"
    with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(midi_path, arcname=Path(midi_path).name)
        zf.write(xml_path, arcname=Path(xml_path).name)

    brief = build_score_brief(scene, genre, emotions, final_prompt)
    journey = emotion_journey(text)
    symbolic_summary = f'''
🎹 **Symbolic Score Parameters**
- Tempo: {params["tempo"]} BPM
- Key: {params["key"]} {params["mode"]}
- Tracks: melody + harmony + bass
- Lead instrument: {params["instrument"]}
- Note preview: {note_preview}

🧪 **Feedback Loop**
- Target emotion: {emotions[0]["label"]}
- Detected from generated audio: {detected_audio_emotion}
- Energy: {features["energy"]:.4f}
- Zero-crossing estimate: {features["zcr"]:.4f}
'''.strip()

    markdown = f"{brief}\n\n🧠 **Emotion Journey**\n{journey}\n\n{symbolic_summary}\n\n💡 *Composer inspiration draft — audio + playable score + editable files.*"

    return (sr, audio), markdown, midi_path, xml_path, str(bundle_path)


In [ ]:

# Quick local test (optional)
# audio_out, md, midi_file, xml_file, bundle_file = generate_scene_to_score(
#     text="She walked slowly through the empty street, remembering what she lost.",
#     duration=10,
#     temperature=1.0,
#     genre="drama",
#     intensity=4,
#     ending="soft fade",
#     leitmotif="memory",
#     use_feedback=True
# )
# print(md)
# print(midi_file, xml_file, bundle_file)


In [ ]:

# -------------------------------------------------
# GRADIO APP
# -------------------------------------------------

with gr.Blocks() as demo:
    gr.Markdown("# 🎬 AI Cinematic Scene-to-Score Generator")
    gr.Markdown(
        "Generate a cinematic audio cue plus a symbolic score that a musician can reuse. "
        "The system produces audio, cue notes, MIDI, MusicXML, and a ZIP bundle."
    )

    with gr.Row():
        text_in = gr.Textbox(
            label="Scene Description",
            lines=6,
            placeholder="Example: He opened the letter with trembling hands. At first he smiled, but then his face fell."
        )
    with gr.Row():
        duration_in = gr.Slider(5, 30, value=10, step=1, label="Audio Duration (seconds)")
        temp_in = gr.Slider(0.7, 1.3, value=1.0, step=0.1, label="Creativity")
    with gr.Row():
        genre_in = gr.Dropdown(["drama", "horror", "romance", "thriller", "sci-fi", "fantasy"], value="drama", label="Film Genre")
        intensity_in = gr.Slider(1, 10, value=5, step=1, label="Scene Intensity")
    with gr.Row():
        ending_in = gr.Dropdown(["soft fade", "resolved", "unresolved", "sudden stop"], value="soft fade", label="Ending Style")
        leitmotif_in = gr.Dropdown(["none", "hero", "villain", "memory", "love"], value="memory", label="Leitmotif")
        feedback_in = gr.Checkbox(value=True, label="Use emotional feedback loop")

    run_btn = gr.Button("Generate Scene-to-Score")

    audio_out = gr.Audio(label="Generated Audio Cue", type="numpy")
    md_out = gr.Markdown(label="Cue Analysis")
    midi_out = gr.File(label="MIDI File")
    xml_out = gr.File(label="MusicXML File")
    zip_out = gr.File(label="ZIP Bundle")

    run_btn.click(
        fn=generate_scene_to_score,
        inputs=[text_in, duration_in, temp_in, genre_in, intensity_in, ending_in, leitmotif_in, feedback_in],
        outputs=[audio_out, md_out, midi_out, xml_out, zip_out]
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6889a761a0340ead01.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
